# SIGNOVA HISTORICAL PIPELINE NOTICE

## KArSL Original Inventory and Metadata Processing

Historical notebook containing archive inspection, video processing, metadata generation and CSV exports.

### Execution safety
- Historical research code: **do not run all cells automatically**.
- Review all Google Drive paths before running individual cells.
- Some cells extract archives and create or overwrite CSV reports.
- Some cells delete temporary directories under `/content`.
- Run only in an isolated Colab runtime with the required source files.
- Confirm sufficient disk space and dependencies before processing.

### Reproducibility limitations
- The code is preserved for provenance, not certified as end-to-end reproducible.
- External datasets, original archives and model assets are not included.
- Stored notebook outputs and execution counts have been removed.
- Full MediaPipe extraction across KArSL-502 was not performed in this project.
- The external Pose dataset is a separate third-party source.
- Source licenses, label approvals and video-to-pose identities remain unresolved.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

drive_root = Path("/content/drive/MyDrive")

print("Drive connected:", drive_root.exists())
print("Files and folders:")

for item in drive_root.iterdir():
    print(item.name)

In [ ]:
from pathlib import Path

data_path = Path("/content/drive/MyDrive/KArSL-502")

print("Folder exists:", data_path.exists())

if data_path.exists():
    for item in list(data_path.iterdir())[:30]:
        print(item.name, "—", "Folder" if item.is_dir() else "File")
else:
    print("Folder not found at this path")

In [ ]:
from pathlib import Path

data_path = Path("/content/drive/MyDrive/KArSL-502")

readme = data_path / "Read Me.txt"

print(readme.read_text(encoding="utf-8", errors="replace"))

In [ ]:
import pandas as pd

labels_path = data_path / "KARSL-502_Labels.xlsx"

excel = pd.ExcelFile(labels_path)

print("Sheets:", excel.sheet_names)

for sheet in excel.sheet_names:
    df = pd.read_excel(labels_path, sheet_name=sheet)

    print(f"\nSheet: {sheet}")
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    display(df.head())

In [ ]:
for folder in ["01", "02", "03"]:
    folder_path = data_path / folder
    items = list(folder_path.rglob("*"))

    files = [p for p in items if p.is_file()]

    print(f"\nFolder {folder}")
    print("Total files:", len(files))
    print("First 10 files:")

    for p in files[:10]:
        print(p.relative_to(data_path))

In [ ]:
import pandas as pd

records = []

for folder in ["01", "02", "03"]:
    for split in ["train", "test"]:
        split_path = data_path / folder / split

        for file in split_path.glob("*.7z"):
            records.append({
                "folder": folder,
                "split": split,
                "file_name": file.name,
                "size_MB": round(file.stat().st_size / (1024**2), 2),
                "path": str(file)
            })

manifest = pd.DataFrame(records)

display(manifest.sort_values(
    ["folder", "split", "file_name"]
))

manifest.to_csv(
    REPORTS_DIR / "download_manifest.csv",
    index=False
)

print("Total archives:", len(manifest))
print("Total size (GB):",
      round(manifest["size_MB"].sum() / 1024, 2))

In [ ]:
for sheet in excel.sheet_names:
    df = pd.read_excel(labels_path, sheet_name=sheet)

    print("\nSheet:", sheet)
    print("Rows and columns:", df.shape)
    print("Column names:", df.columns.tolist())
    print(df.head(10).to_string(index=False))
    print("\nMissing values:")
    print(df.isna().sum().to_string())

In [ ]:
readme_path = data_path / "Read Me.txt"

print(readme_path.read_text(
    encoding="utf-8",
    errors="replace"
))

In [ ]:
from pathlib import Path

REPORTS_DIR = Path(
    "/content/drive/MyDrive/SIGNOVA/KArSL/reports"
)

REPORTS_DIR.mkdir(parents=True, exist_ok=True)

manifest.to_csv(
    REPORTS_DIR / "download_manifest.csv",
    index=False
)

print("Manifest saved to:", REPORTS_DIR / "download_manifest.csv")
print("Number of archives:", len(manifest))
print("Total size (GB):", round(manifest["size_MB"].sum() / 1024, 2))

In [ ]:
import pandas as pd

labels = pd.read_excel(
    data_path / "KARSL-502_Labels.xlsx"
)

print("Data types:")
print(labels.dtypes)

print("\nRandom 20 labels:")
display(labels.sample(min(20, len(labels)), random_state=42))

print("\nLast 10 labels:")
display(labels.tail(10))

print("\nUnique values per column:")
for col in labels.columns:
    print(col, ":", labels[col].nunique())

In [ ]:
import subprocess

archive = data_path / "02" / "test" / "0171-0190.7z"

result = subprocess.run(
    ["7z", "l", str(archive)],
    capture_output=True,
    text=True
)

print(result.stdout[:12000])
print(result.stderr[:2000])

In [ ]:
# Check duplicate labels
print("Duplicate Arabic labels:")
display(labels[labels["Sign-Arabic"].duplicated(keep=False)]
        .sort_values("Sign-Arabic"))

print("Duplicate English labels:")
display(labels[labels["Sign-English"].duplicated(keep=False)]
        .sort_values("Sign-English"))

# Check whether IDs are complete and unique
print("Unique SignIDs:", labels["SignID"].nunique())
print("Missing values:\n", labels.isna().sum())

# Save a clean copy without changing the original
labels.to_csv(
    REPORTS_DIR / "class_mapping.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Saved class_mapping.csv")

In [ ]:
import subprocess
from pathlib import Path

archive = data_path / "02" / "test" / "0171-0190.7z"
sample_dir = Path("/content/karsl_sample")
sample_dir.mkdir(parents=True, exist_ok=True)

# Extract only one video from the archive
result = subprocess.run(
    [
        "7z", "e", str(archive),
        "-o" + str(sample_dir),
        "0171/03_02_0171_(01_05_17_15_51_46)_c.mp4",
        "-y"
    ],
    capture_output=True,
    text=True
)

print("Return code:", result.returncode)
print(result.stdout[-3000:])
print(result.stderr[-1000:])

print("\nExtracted files:")
for f in sample_dir.iterdir():
    print(f.name, round(f.stat().st_size / (1024**2), 2), "MB")

In [ ]:
import cv2
from pathlib import Path

video_path = next(Path("/content/karsl_sample").glob("*.mp4"))

cap = cv2.VideoCapture(str(video_path))

if not cap.isOpened():
    print("ERROR: Cannot open video")
else:
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    print("Video:", video_path.name)
    print("Frames:", frame_count)
    print("FPS:", fps)
    print("Resolution:", width, "x", height)
    print("Duration (seconds):",
          round(frame_count / fps, 2) if fps > 0 else "Unknown")

    success, frame = cap.read()

    if success:
        print("First frame decoded successfully")
        display(__import__("PIL").Image.fromarray(
            cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        ))
    else:
        print("ERROR: Cannot decode first frame")

cap.release()

In [ ]:

import subprocess
import re
import pandas as pd
from pathlib import Path

data_path = Path("/content/drive/MyDrive/KArSL-502")
REPORTS_DIR = Path("/content/drive/MyDrive/SIGNOVA/KArSL/reports")
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

rows = []
errors = []

archives = sorted(data_path.glob("*/*/*.7z"))

print("Archives found:", len(archives))

for i, archive in enumerate(archives, start=1):
    parts = archive.parts
    folder = archive.parent.parent.name
    split = archive.parent.name

    result = subprocess.run(
        ["7z", "l", "-slt", str(archive)],
        capture_output=True,
        text=True
    )

    if result.returncode != 0:
        errors.append({
            "archive": str(archive),
            "error": result.stderr[-500:]
        })
        continue

    current = {}

    for line in result.stdout.splitlines():
        if " = " in line:
            key, value = line.split(" = ", 1)
            current[key] = value

        elif not line.strip():
            name = current.get("Path", "")
            size = current.get("Size", "")

            if name.lower().endswith(".mp4"):
                match = re.search(r"(^|/)(\d{4})/", name)
                sign_id = int(match.group(2)) if match else None

                rows.append({
                    "dataset": "KArSL-502",
                    "folder": folder,
                    "split": split,
                    "archive": archive.name,
                    "video_path_in_archive": name,
                    "sign_id": sign_id,
                    "video_size_bytes": int(size) if size.isdigit() else None,
                    "archive_path": str(archive)
                })

            current = {}

    if i % 5 == 0:
        print(f"Processed {i}/{len(archives)} archives")

metadata = pd.DataFrame(rows)

output_path = REPORTS_DIR / "metadata_karsl.csv"
metadata.to_csv(output_path, index=False, encoding="utf-8-sig")

print("\nVideos listed:", len(metadata))
print("Unique Sign IDs in filenames:", metadata["sign_id"].nunique())
print("Errors:", len(errors))
print("Saved to:", output_path)

if errors:
    display(pd.DataFrame(errors))


In [ ]:

import pandas as pd
from pathlib import Path

REPORTS_DIR = Path("/content/drive/MyDrive/SIGNOVA/KArSL/reports")

metadata = pd.read_csv(REPORTS_DIR / "metadata_karsl.csv")
labels = pd.read_excel(
    "/content/drive/MyDrive/KArSL-502/KARSL-502_Labels.xlsx"
)

# 1. Video counts by folder and split
print("=== Videos by folder and split ===")
display(
    metadata.groupby(["folder", "split"])
    .size()
    .reset_index(name="video_count")
)

# 2. Overall train/test counts
print("\n=== Overall split counts ===")
display(
    metadata.groupby("split")
    .size()
    .reset_index(name="video_count")
)

# 3. Number of videos per sign
counts = (
    metadata.groupby("sign_id")
    .size()
    .reindex(labels["SignID"], fill_value=0)
)

print("\n=== Videos per sign ===")
print("Signs with zero videos:", int((counts == 0).sum()))
print("Minimum videos per sign:", int(counts.min()))
print("Maximum videos per sign:", int(counts.max()))
print("Average videos per sign:", round(counts.mean(), 2))

# 4. Check duplicate paths
print(
    "\nDuplicate video paths:",
    int(metadata["video_path_in_archive"].duplicated().sum())
)

# 5. Check IDs missing from the labels
metadata_ids = set(metadata["sign_id"].dropna().astype(int))
label_ids = set(labels["SignID"].astype(int))

print("IDs in videos but not labels:", sorted(metadata_ids - label_ids))
print("IDs in labels but not videos:", sorted(label_ids - metadata_ids))

# 6. Save a basic quality-control report
report_path = REPORTS_DIR / "qc_report.md"

with open(report_path, "w", encoding="utf-8") as f:
    f.write("# KArSL-502 Preliminary QC Report\n\n")
    f.write(f"- Videos listed in archives: {len(metadata)}\n")
    f.write(f"- Unique sign IDs in filenames: {metadata['sign_id'].nunique()}\n")
    f.write(f"- Signs with zero videos: {int((counts == 0).sum())}\n")
    f.write(f"- Minimum videos per sign: {int(counts.min())}\n")
    f.write(f"- Maximum videos per sign: {int(counts.max())}\n")
    f.write(f"- Duplicate video paths: {int(metadata['video_path_in_archive'].duplicated().sum())}\n")

print("\nSaved preliminary report:", report_path)


In [ ]:

import pandas as pd
from pathlib import Path

REPORTS_DIR = Path(
    "/content/drive/MyDrive/SIGNOVA/KArSL/reports"
)

metadata = pd.read_csv(REPORTS_DIR / "metadata_karsl.csv")

# Number of videos for each sign in each split
distribution = (
    metadata.groupby(["sign_id", "split"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=["train", "test"], fill_value=0)
)

distribution["total"] = distribution["train"] + distribution["test"]

print("Signs missing from train:", int((distribution["train"] == 0).sum()))
print("Signs missing from test:", int((distribution["test"] == 0).sum()))

print("\nTrain videos per sign:")
print(distribution["train"].describe())

print("\nTest videos per sign:")
print(distribution["test"].describe())

print("\nSigns with the fewest total videos:")
display(distribution.sort_values("total").head(10))

print("\nSigns with the most total videos:")
display(distribution.sort_values("total", ascending=False).head(10))

# Save detailed distribution
distribution.to_csv(
    REPORTS_DIR / "sign_split_distribution.csv",
    index_label="SignID"
)

print("\nSaved sign_split_distribution.csv")


In [ ]:
import subprocess
import pandas as pd
from pathlib import Path

DATASET_DIR = Path("/content/drive/MyDrive/KArSL-502")
REPORTS_DIR = Path("/content/drive/MyDrive/SIGNOVA/KArSL/reports")
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

archives = sorted(DATASET_DIR.rglob("*.7z"))
results = []

print(f"Found {len(archives)} archives")

for i, archive in enumerate(archives, start=1):
    print(f"[{i}/{len(archives)}] Testing: {archive.name}")

    result = subprocess.run(
        ["7z", "t", str(archive)],
        capture_output=True,
        text=True
    )

    output = result.stdout + "\n" + result.stderr

    results.append({
        "archive": str(archive),
        "status": "PASS" if result.returncode == 0 else "FAIL",
        "return_code": result.returncode,
        "output": output[-2000:]
    })

    print("Status:", results[-1]["status"])

integrity_df = pd.DataFrame(results)
output_path = REPORTS_DIR / "archive_integrity.csv"
integrity_df.to_csv(output_path, index=False, encoding="utf-8-sig")

print("\n--- Summary ---")
print(integrity_df["status"].value_counts())
print(f"Report saved to: {output_path}")

In [ ]:
import pandas as pd
import cv2
import subprocess
import random
from pathlib import Path

REPORTS_DIR = Path("/content/drive/MyDrive/SIGNOVA/KArSL/reports")

metadata = pd.read_csv(REPORTS_DIR / "metadata_karsl.csv")

# Select 30 videos across train/test and dataset folders
sample = (
    metadata.groupby(["folder", "split"], group_keys=False)
    .sample(n=5, random_state=42)
    .reset_index(drop=True)
)

results = []

for i, row in sample.iterrows():
    archive = row["archive_path"]
    video_path = row["video_path_in_archive"]

    # Extract only this video into a temporary folder
    temp_dir = Path("/content/karsl_qc_sample") / str(i)
    temp_dir.mkdir(parents=True, exist_ok=True)

    result = subprocess.run(
        ["7z", "e", "-y", f"-o{temp_dir}", archive, video_path],
        capture_output=True,
        text=True
    )

    videos = list(temp_dir.glob("*.mp4"))

    status = "EXTRACTION_FAILED"
    frames = 0
    fps = 0

    if result.returncode == 0 and videos:
        cap = cv2.VideoCapture(str(videos[0]))
        if cap.isOpened():
            frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            fps = cap.get(cv2.CAP_PROP_FPS)
            status = "PASS" if frames > 0 and fps > 0 else "DECODE_FAILED"
        cap.release()

    results.append({
        "archive": archive,
        "video_path": video_path,
        "status": status,
        "frames": frames,
        "fps": fps
    })

qc_df = pd.DataFrame(results)
qc_df.to_csv(
    REPORTS_DIR / "video_sample_qc.csv",
    index=False,
    encoding="utf-8-sig"
)

print(qc_df["status"].value_counts())
print("\nFailed videos:")
display(qc_df[qc_df["status"] != "PASS"])
print("\nSaved video_sample_qc.csv")

In [ ]:
from pathlib import Path
import pandas as pd
from datetime import datetime

REPORTS_DIR = Path("/content/drive/MyDrive/SIGNOVA/KArSL/reports")

metadata = pd.read_csv(REPORTS_DIR / "metadata_karsl.csv")
integrity = pd.read_csv(REPORTS_DIR / "archive_integrity.csv")
video_qc = pd.read_csv(REPORTS_DIR / "video_sample_qc.csv")
distribution = pd.read_csv(REPORTS_DIR / "sign_split_distribution.csv")

total_videos = len(metadata)
total_signs = metadata["sign_id"].nunique()
train_count = (metadata["split"] == "train").sum()
test_count = (metadata["split"] == "test").sum()

archive_pass = (integrity["status"] == "PASS").sum()
video_pass = (video_qc["status"] == "PASS").sum()

report = f"""# KArSL-502 Quality Control Report

## Dataset overview
- Total videos listed: {total_videos}
- Unique sign classes: {total_signs}
- Train videos: {train_count}
- Test videos: {test_count}

## Archive integrity
- Archives tested: {len(integrity)}
- Archives passed: {archive_pass}
- Archives failed: {(integrity["status"] == "FAIL").sum()}

## Sample video validation
- Videos sampled: {len(video_qc)}
- Videos passed: {video_pass}
- Videos failed: {(video_qc["status"] != "PASS").sum()}

## Class coverage
- Signs missing from train: {(distribution["train"] == 0).sum()}
- Signs missing from test: {(distribution["test"] == 0).sum()}

## Limitations
- Only a sample of videos was decoded and checked.
- Passing archive integrity tests does not guarantee every video is decodable.
- Translation review candidates must be finalized before producing the corrected label mapping.
- No model training or evaluation has been performed at this stage.

## Report generated
{datetime.now().strftime("%Y-%m-%d %H:%M")}
"""

report_path = REPORTS_DIR / "qc_report.md"
report_path.write_text(report, encoding="utf-8")

print(report)
print(f"\nUpdated report: {report_path}")

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

base = Path("/content/drive/MyDrive/SIGNOVA/KArSL/reports")

print("Reports folder exists:", base.exists())

if base.exists():
    print("\nFiles found:")
    for file in sorted(base.iterdir()):
        print(file.name)
else:
    print("SIGNOVA/KArSL/reports folder was not found.")

In [ ]:
import pandas as pd
from pathlib import Path
from datetime import datetime

REPORTS_DIR = Path("/content/drive/MyDrive/SIGNOVA/KArSL/reports")

# Read the existing reports
files = {
    "metadata": "metadata_karsl.csv",
    "integrity": "archive_integrity.csv",
    "video_qc": "video_sample_qc.csv",
    "distribution": "sign_split_distribution.csv",
}

data = {}

for key, filename in files.items():
    path = REPORTS_DIR / filename
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    data[key] = pd.read_csv(path)
    print(f"Loaded {filename}: {len(data[key])} rows")

metadata = data["metadata"]
integrity = data["integrity"]
video_qc = data["video_qc"]
distribution = data["distribution"]

report = f"""# KArSL-502 Quality Control Report

## Dataset overview
- Videos listed: {len(metadata)}
- Unique sign IDs: {metadata['sign_id'].nunique()}
- Train videos: {(metadata['split'] == 'train').sum()}
- Test videos: {(metadata['split'] == 'test').sum()}

## Archive integrity
- Archives tested: {len(integrity)}
- Passed: {(integrity['status'] == 'PASS').sum()}
- Failed: {(integrity['status'] != 'PASS').sum()}

## Sample video validation
- Videos sampled: {len(video_qc)}
- Passed: {(video_qc['status'] == 'PASS').sum()}
- Failed: {(video_qc['status'] != 'PASS').sum()}

## Class coverage
- Signs missing from train: {(distribution['train'] == 0).sum()}
- Signs missing from test: {(distribution['test'] == 0).sum()}

## Limitations
- Only 30 sample videos were decoded and checked.
- Archive integrity does not guarantee that every video can be decoded.
- Translation corrections still need to be finalized in a separate label file.
- Model training and evaluation have not been performed.

## Last updated
{datetime.now().strftime("%Y-%m-%d %H:%M")}
"""

report_path = REPORTS_DIR / "qc_report.md"
report_path.write_text(report, encoding="utf-8")

print("\nQC report updated successfully:")
print(report_path)
print("\n", report)

In [ ]:
from pathlib import Path
import pandas as pd
from datetime import datetime

REPORTS_DIR = Path("/content/drive/MyDrive/SIGNOVA/KArSL/reports")

metadata = pd.read_csv(REPORTS_DIR / "metadata_karsl.csv")
integrity = pd.read_csv(REPORTS_DIR / "archive_integrity.csv")
video_qc = pd.read_csv(REPORTS_DIR / "video_sample_qc.csv")
distribution = pd.read_csv(REPORTS_DIR / "sign_split_distribution.csv")

labels_path = Path("/content/drive/MyDrive/KArSL-502/KARSL-502_Labels.xlsx")
labels = pd.read_excel(labels_path)

total_videos = len(metadata)
total_signs = metadata["sign_id"].nunique()
train_count = (metadata["split"] == "train").sum()
test_count = (metadata["split"] == "test").sum()

dataset_card = f"""# KArSL-502 Dataset Card

## Overview
This dataset preparation covers KArSL-502 for the SIGNOVA project.

## Dataset statistics
- Video entries listed in archives: {total_videos}
- Unique sign IDs found in video filenames: {total_signs}
- Train videos: {train_count}
- Test videos: {test_count}
- Rows in label spreadsheet: {len(labels)}

## Archive integrity
- Archives tested: {len(integrity)}
- Archives passed: {(integrity["status"] == "PASS").sum()}
- Archives failed: {(integrity["status"] != "PASS").sum()}

## Sample video validation
- Videos decoded and checked: {len(video_qc)}
- Videos passed: {(video_qc["status"] == "PASS").sum()}
- Videos failed: {(video_qc["status"] != "PASS").sum()}

## Class coverage
- Sign IDs missing from train: {(distribution["train"] == 0).sum()}
- Sign IDs missing from test: {(distribution["test"] == 0).sum()}

## Label notes
- Translation review candidates were prepared separately.
- Translation corrections must be confirmed before using a corrected mapping for training.
- The original label spreadsheet is preserved.

## Limitations
- Only a sample of videos was decoded; the entire collection has not been decoded individually.
- No model training or evaluation has been performed.
- The meaning of archive folder names and filename components has not been established.

## Prepared
{datetime.now().strftime("%Y-%m-%d")}
"""

handoff = f"""# SIGNOVA KArSL-502 Data Handoff

## Current status
Dataset inventory and initial quality checks are complete.

## Files prepared
- metadata_karsl.csv
- download_manifest.csv
- class_mapping.csv
- sign_split_distribution.csv
- archive_integrity.csv
- video_sample_qc.csv
- qc_report.md
- dataset_card.md

## Key counts
- Video entries: {total_videos}
- Sign classes: {total_signs}
- Train: {train_count}
- Test: {test_count}
- Archives passed: {(integrity["status"] == "PASS").sum()}/{len(integrity)}
- Sample videos passed: {(video_qc["status"] == "PASS").sum()}/{len(video_qc)}

## Before model training
1. Finalize and approve corrected English labels in a separate file.
2. Confirm the intended evaluation split and check for possible subject/session leakage if relevant metadata is available.
3. Expand video decoding checks if required by the project.
4. Choose the modeling approach and preprocessing pipeline.
5. Keep the original labels and raw archives unchanged.

## Important
Passing archive checks and a 30-video sample check do not prove that every video is valid.
This handoff documents data preparation only; it does not claim that a model is trained.
"""

(REPORTS_DIR / "dataset_card.md").write_text(dataset_card, encoding="utf-8")
(REPORTS_DIR / "handoff.md").write_text(handoff, encoding="utf-8")

print("Created:")
print(REPORTS_DIR / "dataset_card.md")
print(REPORTS_DIR / "handoff.md")
print("\nDataset Card and Handoff created successfully.")

In [ ]:

from google.colab import files
import pandas as pd
import io

uploaded = files.upload()

# قراءة الملف المرفوع
filename = next(iter(uploaded))
labels = pd.read_csv(
    io.BytesIO(uploaded[filename]),
    encoding="utf-8-sig"
)

print("Loaded file:", filename)
print("Number of labels:", len(labels))
print("Columns:", labels.columns.tolist())

display(labels.head())


In [ ]:

import os
import pandas as pd

METADATA_PATH = (
    "/content/drive/MyDrive/SIGNOVA/KArSL/"
    "reports/metadata_karsl.csv"
)

metadata = pd.read_csv(METADATA_PATH)

# توحيد نوع SignID
metadata["SignID"] = metadata["SignID"].astype(str).str.strip()
labels["SignID"] = labels["SignID"].astype(str).str.strip()

# ربط الفيديوهات بالترجمات الجديدة
linked_data = metadata.merge(
    labels[["SignID", "Sign-Arabic", "Sign-English"]],
    on="SignID",
    how="left",
    validate="many_to_one",
    indicator=True
)

print("Total videos:", len(linked_data))
print("Videos linked:", (linked_data["_merge"] == "both").sum())
print("Videos without labels:", (linked_data["_merge"] == "left_only").sum())

display(
    linked_data[
        ["SignID", "Sign-Arabic", "Sign-English", "_merge"]
    ].head(10)
)


In [ ]:

print("Metadata columns:")
print([repr(col) for col in metadata.columns])

print("\nLabels columns:")
print([repr(col) for col in labels.columns])

print("\nMetadata preview:")
display(metadata.head(3))

print("\nLabels preview:")
display(labels.head(3))


In [ ]:

print("Metadata columns:")
print([repr(col) for col in metadata.columns])

print("\nLabels columns:")
print([repr(col) for col in labels.columns])

print("\nMetadata preview:")
display(metadata.head(3))

print("\nLabels preview:")
display(labels.head(3))


In [ ]:

import os
import pandas as pd
from google.colab import files
import io

# 1. قراءة ملف الـ metadata
METADATA_PATH = (
    "/content/drive/MyDrive/SIGNOVA/KArSL/"
    "reports/metadata_karsl.csv"
)

if not os.path.exists(METADATA_PATH):
    print("Metadata file not found:", METADATA_PATH)
else:
    metadata = pd.read_csv(METADATA_PATH)
    print("Metadata loaded successfully!")
    print("Metadata columns:")
    print([repr(col) for col in metadata.columns])
    display(metadata.head(3))

# 2. التأكد من وجود ملف الـ Labels في الجلسة
if "labels" in globals():
    print("\nLabels are already loaded.")
    print("Labels columns:", [repr(col) for col in labels.columns])
else:
    print("\nPlease upload the reviewed labels CSV.")
    uploaded = files.upload()

    if uploaded:
        filename = next(iter(uploaded))
        labels = pd.read_csv(
            io.BytesIO(uploaded[filename]),
            encoding="utf-8-sig"
        )
        print("Labels loaded:", filename)
        print("Labels columns:", [repr(col) for col in labels.columns])
        display(labels.head(3))


In [ ]:

from google.colab import drive
import os
import pandas as pd

# Mount Google Drive
drive.mount("/content/drive")

# Search for metadata file
search_roots = [
    "/content/drive/MyDrive/SIGNOVA",
    "/content/drive/MyDrive/KArSL-502"
]

print("Searching for metadata_karsl.csv ...")

for root in search_roots:
    if os.path.exists(root):
        for current_root, dirs, files in os.walk(root):
            if "metadata_karsl.csv" in files:
                print("FOUND:", os.path.join(
                    current_root, "metadata_karsl.csv"
                ))

# Inspect the uploaded labels
print("\nLabels preview:")
print(labels.head(10).to_string(index=False))

print("\nData types:")
print(labels.dtypes)

print("\nUnique values in first rows:")
print(labels.head(10).to_dict("records"))


In [ ]:

import os
import pandas as pd

# Original labels file
original_path = (
    "/content/drive/MyDrive/KArSL-502/"
    "KARSL-502_Labels.xlsx"
)

# Uploaded reviewed labels file
reviewed_path = (
    "/content/SIGNOVA_KArSL_labels_reviewed_draft (1).csv"
)

print("Original file exists:", os.path.exists(original_path))
print("Reviewed file exists:", os.path.exists(reviewed_path))

if os.path.exists(original_path):
    original_labels = pd.read_excel(original_path)
    print("\nOriginal labels:")
    display(original_labels.head(10))

if os.path.exists(reviewed_path):
    reviewed_labels = pd.read_csv(reviewed_path, encoding="utf-8-sig")
    print("\nReviewed labels:")
    display(reviewed_labels.head(10))


In [ ]:

import pandas as pd

original_path = (
    "/content/drive/MyDrive/KArSL-502/"
    "KARSL-502_Labels.xlsx"
)

# عرض أسماء كل الـSheets
excel_file = pd.ExcelFile(original_path)
print("Sheet names:", excel_file.sheet_names)

# فحص أول 5 صفوف من كل Sheet
for sheet in excel_file.sheet_names:
    print(f"\n--- Sheet: {sheet} ---")

    df = pd.read_excel(
        original_path,
        sheet_name=sheet,
        header=None,
        nrows=5
    )

    display(df)


In [ ]:
# عرض أول 15 صف
display(labels.head(15))

# عرض صفوف من قائمة المراجعة، لو SignID موجودة
review_ids = [72, 75, 78, 95, 96, 364, 391]

display(labels[labels["SignID"].isin(review_ids)])

In [ ]:
import pandas as pd

METADATA_PATH = (
    "/content/drive/MyDrive/SIGNOVA/KArSL/"
    "reports/metadata_karsl.csv"
)

metadata = pd.read_csv(METADATA_PATH)

# تنظيف أسماء الأعمدة من المسافات
metadata.columns = metadata.columns.str.strip()
labels.columns = labels.columns.str.strip()

print("Metadata columns:", metadata.columns.tolist())

In [ ]:

# توحيد أسماء الأعمدة وأنواع الـIDs
metadata.columns = metadata.columns.str.strip()
labels.columns = labels.columns.str.strip()

metadata["sign_id"] = metadata["sign_id"].astype(str).str.strip()
labels["SignID"] = labels["SignID"].astype(str).str.strip()

# التأكد من عدم وجود IDs مكررة في ملف الـLabels
assert labels["SignID"].is_unique, "Duplicate SignID in labels!"

# ربط كل فيديو بالترجمة العربية والإنجليزية
linked_data = metadata.merge(
    labels[["SignID", "Sign-Arabic", "Sign-English"]],
    left_on="sign_id",
    right_on="SignID",
    how="left",
    validate="many_to_one",
    indicator=True
)

# عرض نتائج الربط
print("Total videos:", len(linked_data))
print("Videos linked:", (linked_data["_merge"] == "both").sum())
print("Videos without labels:", (linked_data["_merge"] == "left_only").sum())

# عرض عينة
display(
    linked_data[
        ["sign_id", "video_path_in_archive",
         "Sign-Arabic", "Sign-English", "_merge"]
    ].head(10)
)


In [ ]:

# Check reviewed translations
check_ids = ["72", "75", "78", "95", "96", "364", "391"]

labels["SignID"] = labels["SignID"].astype(str).str.strip()

check = labels.loc[
    labels["SignID"].isin(check_ids),
    ["SignID", "Sign-Arabic", "Sign-English"]
].sort_values("SignID")

print("Reviewed labels check:")
display(check)

print("\nTotal labels:", len(labels))
print("Unique SignIDs:", labels["SignID"].nunique())
print("Missing reviewed IDs:", set(check_ids) - set(labels["SignID"]))


In [ ]:

# Check that reviewed labels exist in linked metadata
linked_check = linked_data.loc[
    linked_data["sign_id"].astype(str).isin(check_ids),
    ["sign_id", "Sign-Arabic", "Sign-English"]
].drop_duplicates().sort_values("sign_id")

display(linked_check)

print("Total videos:", len(linked_data))
print(
    "Videos without labels:",
    linked_data["Sign-English"].isna().sum()
)


In [ ]:

import os

output_path = (
    "/content/drive/MyDrive/SIGNOVA/KArSL/reports/"
    "linked_metadata_reviewed.csv"
)

os.makedirs(os.path.dirname(output_path), exist_ok=True)

# Remove technical merge indicator and redundant ID column
final_metadata = linked_data.drop(
    columns=["_merge", "SignID"],
    errors="ignore"
)

final_metadata.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved successfully:", output_path)
print("Rows:", len(final_metadata))
print("Columns:", final_metadata.columns.tolist())
print("File size (MB):", round(os.path.getsize(output_path) / (1024**2), 2))


In [ ]:

import pandas as pd

metadata_path = (
    "/content/drive/MyDrive/SIGNOVA/KArSL/reports/"
    "linked_metadata_reviewed.csv"
)

df = pd.read_csv(metadata_path)

print("=== Dataset Overview ===")
print("Total videos:", len(df))
print("Unique sign IDs:", df["sign_id"].nunique())
print("Missing English labels:", df["Sign-English"].isna().sum())
print("Missing Arabic labels:", df["Sign-Arabic"].isna().sum())
print("Duplicate video paths:", df["video_path_in_archive"].duplicated().sum())

print("\n=== Videos per split ===")
print(df["split"].value_counts(dropna=False))

print("\n=== Unique signs per split ===")
print(df.groupby("split")["sign_id"].nunique())

print("\n=== Signs missing from either split ===")
signs_per_split = df.groupby("sign_id")["split"].nunique()
print("Signs not present in all splits:", (signs_per_split < df["split"].nunique()).sum())


In [ ]:

import pandas as pd

path = (
    "/content/drive/MyDrive/SIGNOVA/KArSL/reports/"
    "linked_metadata_reviewed.csv"
)

df = pd.read_csv(path)

# Number of videos for each sign
class_distribution = (
    df.groupby(["sign_id", "Sign-Arabic", "Sign-English"])
      .size()
      .reset_index(name="video_count")
      .sort_values("video_count")
)

print("Number of classes:", len(class_distribution))
print("\n=== Dataset Statistics ===")
print(class_distribution["video_count"].describe())

print("\n=== 10 Classes with Fewest Videos ===")
display(class_distribution.head(10))

print("\n=== 10 Classes with Most Videos ===")
display(class_distribution.tail(10).sort_values(
    "video_count", ascending=False
))


In [ ]:

# Count videos per class in each split
split_distribution = (
    df.groupby(["sign_id", "split"])
      .size()
      .unstack(fill_value=0)
)

print("=== Train Videos per Class ===")
print(split_distribution["train"].describe())

print("\n=== Test Videos per Class ===")
print(split_distribution["test"].describe())

print("\n=== Classes with the Fewest Train Samples ===")
display(split_distribution.sort_values("train").head(10))

print("\n=== Classes with the Fewest Test Samples ===")
display(split_distribution.sort_values("test").head(10))

print("\n=== Total Videos per Split ===")
print(split_distribution.sum())


In [ ]:

from google.colab import drive
drive.mount("/content/drive")
import pandas as pd

# Load the saved dataset metadata
path = (
    "/content/drive/MyDrive/SIGNOVA/KArSL/reports/"
    "linked_metadata_reviewed.csv"
)

df = pd.read_csv(path)

# Count videos per class in each split
split_distribution = (
    df.groupby(["sign_id", "split"])
      .size()
      .unstack(fill_value=0)
)

print("=== Train Videos per Class ===")
print(split_distribution["train"].describe())

print("\n=== Test Videos per Class ===")
print(split_distribution["test"].describe())

print("\n=== Classes with the Fewest Train Samples ===")
display(split_distribution.sort_values("train").head(10))

print("\n=== Classes with the Fewest Test Samples ===")
display(split_distribution.sort_values("test").head(10))

print("\n=== Total Videos per Split ===")
print(split_distribution.sum())


In [ ]:
import pandas as pd
df = pd.read_csv("/content/drive/MyDrive/SIGNOVA/KArSL/reports/linked_metadata_reviewed.csv")
name = df.video_path_in_archive.str.split("/").str[-1]
x = name.str.extract(r"^(\d+)_(\d+)_(\d{4})_\((\d+)_(\d+)_(\d+)_(\d+)_(\d+)_(\d+)\)_c\.mp4$")
x.columns = ["f1","f2","sign","dd","mm","yy","HH","MI","SS"]
print("unparsed:", x.f1.isna().sum())
df = pd.concat([df, x], axis=1)
df["date"] = df.dd + "-" + df.mm + "-" + df.yy
print(pd.crosstab(df.folder, df.f1))
print(pd.crosstab(df.folder, df.f2))
print(df.groupby(["folder","f1","f2"]).date.nunique())
print("sign mismatch:", (df.sign.astype(int) != df.sign_id.astype(int)).sum())
print("duplicate filenames:", name.duplicated().sum())

In [ ]:
df["signer_id"] = df.f2
df["date"] = df.dd + "-" + df.mm + "-" + df.yy

# 1) الـ 33 فيديو المشبوهين
mm = df[df.sign.astype(int) != df.sign_id.astype(int)]
display(mm[["signer_id","split","archive","sign_id","sign","video_path_in_archive"]])

# 2) أي (مترجم، إشارة) عددها مش 50
c = df.groupby(["signer_id","sign_id"]).size().unstack(0)
dev = c[(c != 50).any(axis=1)]
print("signs where count != 50 for any signer:", len(dev))
display(dev.head(40))

# 3) هل نفس اليوم/الجلسة موجود في train وtest؟
t = pd.crosstab([df.signer_id, df.f1, df.date], df.split)
both = t[(t.train > 0) & (t.test > 0)]
print("session-days:", len(t), "| appearing in BOTH train and test:", len(both))

In [ ]:
import pandas as pd
R = "/content/drive/MyDrive/SIGNOVA/KArSL/reports/"
df = pd.read_csv(R + "linked_metadata_reviewed.csv")
name = df.video_path_in_archive.str.split("/").str[-1]
x = name.str.extract(r"^(\d+)_(\d+)_(\d{4})_\((\d+)_(\d+)_(\d+)_(\d+)_(\d+)_(\d+)\)_c\.mp4$")
x.columns = ["batch","signer","name_sign","dd","mm","yy","HH","MI","SS"]
df = pd.concat([df.drop(columns=["folder"]), x], axis=1)
df = df.sort_values(["archive_path","video_path_in_archive"]).reset_index(drop=True)

df["sample_id"] = ["KARSL_%06d" % i for i in range(1, len(df)+1)]
df["signer_id"] = "S" + df.signer
df["recording_date"] = df.yy + "-" + df.mm + "-" + df.dd
df["session_id"] = df.signer_id + "_B" + df.batch + "_" + df.recording_date  # مشتق من اسم الملف

df["qc_status"] = "PENDING"   # لسه ما اتفحصش decode
df["qc_flags"] = ""
m = df.name_sign.astype(int) != df.sign_id.astype(int)
df.loc[m, ["qc_status","qc_flags"]] = ["REVIEW","FILENAME_SIGN_MISMATCH"]

c = df.groupby(["signer_id","sign_id"]).size().rename("n").reset_index()
bad = c[c.n != 50]
key = df.set_index(["signer_id","sign_id"]).index.isin(bad.set_index(["signer_id","sign_id"]).index)
df.loc[key & (df.qc_flags == ""), "qc_flags"] = "COUNT_NOT_50"

df.to_csv(R + "metadata_karsl_canonical.csv", index=False, encoding="utf-8-sig")
df[df.qc_flags != ""][["sample_id","qc_status","qc_flags","signer_id","sign_id","video_path_in_archive"]] \
  .to_csv(R + "qc_flags.csv", index=False, encoding="utf-8-sig")

sm = df[["sample_id","signer_id","session_id","sign_id","split"]].rename(columns={"split":"author_split"})
for s in ["S01","S02","S03"]:
    sm["loso_test_"+s] = (df.signer_id == s).map({True:"test", False:"train"})
sm["split_version"] = "v1"
sm.to_csv(R + "split_manifest.csv", index=False)

print(df.qc_status.value_counts(), "\n", df.qc_flags.value_counts(), "\n", df.signer_id.value_counts())

In [ ]:
R = "/content/drive/MyDrive/SIGNOVA/KArSL/reports/"
df = pd.read_csv(R + "metadata_karsl_canonical.csv", dtype={"signer":str})
df.loc[df.qc_flags == "COUNT_NOT_50", "qc_flags"] = ""

c = df.groupby(["signer_id","sign_id"]).size().rename("n").reset_index()
c["expected"] = 50; c["diff"] = c.n - 50
c[c["diff"] != 0].sort_values("diff").to_csv(R + "count_anomalies.csv", index=False)
print(c[c["diff"].abs() >= 10])          # الحالات الكبيرة بس

df.to_csv(R + "metadata_karsl_canonical.csv", index=False, encoding="utf-8-sig")
df[df.qc_flags != ""][["sample_id","qc_status","qc_flags","signer_id","sign_id","video_path_in_archive"]] \
  .to_csv(R + "qc_flags.csv", index=False, encoding="utf-8-sig")

In [ ]:
R = "/content/drive/MyDrive/SIGNOVA/KArSL/reports/"
df = pd.read_csv(R + "metadata_karsl_canonical.csv", dtype={"signer": str})
d = df[(df.signer_id == "S01") & df.sign_id.isin([1,5,6,7,27])].copy()
d["t"] = pd.to_datetime(dict(year=2000+d.yy, month=d.mm, day=d.dd,
                             hour=d.HH, minute=d.MI, second=d.SS))
print(d.groupby(["sign_id","name_sign","split"]).agg(
    n=("t","size"), first=("t","min"), last=("t","max")))

x = d[d.sign_id == 27].sort_values("t").copy()
x["gap"] = x.t.diff()
print(x.nlargest(3, "gap")[["t","gap","split"]])

In [ ]:
R = "/content/drive/MyDrive/SIGNOVA/KArSL/reports/"
df = pd.read_csv(R + "metadata_karsl_canonical.csv", dtype={"signer": str}, low_memory=False)

d = df[(df.signer_id == "S01") & df.sign_id.isin([5,6,7])]
print(d.groupby(["sign_id","recording_date","name_sign"]).size())

x = df[(df.signer_id == "S01") & (df.sign_id == 27)].copy()
x["t"] = pd.to_datetime(dict(year=2000+x.yy, month=x.mm, day=x.dd, hour=x.HH, minute=x.MI, second=x.SS))
x = x.sort_values("t")
print(x.t.diff().dt.total_seconds().value_counts().head(8))
print("duplicate sizes:", x.video_size_bytes.duplicated().sum())

In [ ]:
R = "/content/drive/MyDrive/SIGNOVA/KArSL/reports/"
df = pd.read_csv(R + "metadata_karsl_canonical.csv", dtype={"signer": str}, low_memory=False)
d = df[(df.signer_id=="S01") & (df.sign_id==5) & (df.recording_date=="16-11-10")].copy()
d["t"] = pd.to_datetime(dict(year=2000+d.yy, month=d.mm, day=d.dd, hour=d.HH, minute=d.MI, second=d.SS))
d = d.sort_values("t")
g = d.t.diff().dt.total_seconds()
d["cluster"] = (g > 10).cumsum()
print(d.groupby("cluster").agg(n=("t","size"), start=("t","min"), end=("t","max")))
print(g[g > 10])

In [ ]:
import subprocess, cv2, matplotlib.pyplot as plt
from pathlib import Path
tmp = Path("/content/chk"); tmp.mkdir(exist_ok=True)
fig, axs = plt.subplots(1, d.cluster.nunique(), figsize=(6*d.cluster.nunique(), 4))
axs = [axs] if d.cluster.nunique() == 1 else axs
for ax, (cl, grp) in zip(axs, d.groupby("cluster")):
    r = grp.iloc[len(grp)//2]
    subprocess.run(["7z","e",r.archive_path,f"-o{tmp}",r.video_path_in_archive,"-y","-bso0","-bsp0"], check=True)
    cap = cv2.VideoCapture(str(tmp / Path(r.video_path_in_archive).name))
    cap.set(cv2.CAP_PROP_POS_FRAMES, cap.get(cv2.CAP_PROP_FRAME_COUNT)//2)
    ok, f = cap.read(); cap.release()
    ax.imshow(cv2.cvtColor(f, cv2.COLOR_BGR2RGB)); ax.set_title(f"cluster {cl} (n={len(grp)})"); ax.axis("off")
plt.show()

In [ ]:
R = "/content/drive/MyDrive/SIGNOVA/KArSL/reports/"
df = pd.read_csv(R + "metadata_karsl_canonical.csv", dtype={"signer": str}, low_memory=False)

df["t"] = pd.to_datetime(dict(year=2000+df.yy.astype(int), month=df.mm.astype(int), day=df.dd.astype(int),
                              hour=df.HH.astype(int), minute=df.MI.astype(int), second=df.SS.astype(int)))
m = ((df.signer_id=="S01") & (df.sign_id==5) &
     (df.t >= "2016-11-10 16:34:00") & (df.t <= "2016-11-10 16:35:10"))
print("flagged:", m.sum())                      # لازم يطلع 33

df["sign_id_original"] = df.sign_id
df["sign_id_corrected"] = df.sign_id
df.loc[m, "sign_id_corrected"] = 6              # بعد التأكيد البصري بس
df.loc[m, ["qc_status","qc_flags"]] = ["REVIEW","SUSPECTED_MISFILED_SIGN6"]
df = df.drop(columns=["t"])
df.to_csv(R + "metadata_karsl_canonical.csv", index=False, encoding="utf-8-sig")

In [ ]:
import subprocess, cv2, matplotlib.pyplot as plt
from pathlib import Path
R = "/content/drive/MyDrive/SIGNOVA/KArSL/reports/"
df = pd.read_csv(R + "metadata_karsl_canonical.csv", dtype={"signer": str}, low_memory=False)
tmp = Path("/content/chk"); tmp.mkdir(exist_ok=True)

groups = {
  "cluster0 (sign 5?)": df[(df.signer_id=="S01")&(df.sign_id_original==5)&(df.qc_flags!="SUSPECTED_MISFILED_SIGN6")&(df.recording_date=="16-11-10")],
  "cluster1 (sign 6?)": df[df.qc_flags=="SUSPECTED_MISFILED_SIGN6"],
}
fig, axs = plt.subplots(2, 3, figsize=(15, 7))
for row, (title, g) in zip(axs, groups.items()):
    r = g.iloc[len(g)//2]
    subprocess.run(["7z","e",r.archive_path,f"-o{tmp}",r.video_path_in_archive,"-y","-bso0","-bsp0"], check=True)
    cap = cv2.VideoCapture(str(tmp / Path(r.video_path_in_archive).name))
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    for ax, frac in zip(row, [0.35, 0.5, 0.65]):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(n*frac)); ok, f = cap.read()
        ax.imshow(cv2.cvtColor(f, cv2.COLOR_BGR2RGB)); ax.set_title(f"{title} @ {int(frac*100)}%"); ax.axis("off")
    cap.release()
plt.tight_layout(); plt.show()